# 06 — Segmentasi Pelanggan: K-Means (Bab 2.4b)Notebook ini mengimplementasikan **segmentasi pelanggan** menggunakan algoritma **K-Means Clustering** dari Spark MLlib.

## 6.1 Inisialisasi & Load Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum

spark = SparkSession.builder \
    .appName("06_KMeans_Segmentation") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

df = spark.read.parquet("/output/retail_parquet")
print(f"✅ Loaded {df.count()} rows")

## 6.2 Feature EngineeringBuat fitur per pelanggan: frequency, monetary, avg_quantity

In [ ]:
# Agregasi fitur per pelanggan
df_customer = df.groupBy("Customer_ID").agg(
    count("*").alias("frequency"),
    avg("Total_Amount").alias("avg_monetary"),
    avg("Quantity").alias("avg_quantity"),
    spark_sum("Total_Amount").alias("total_spending")
)

print(f"✅ {df_customer.count()} pelanggan unik")
df_customer.show(10)
df_customer.describe().show()

## 6.3 Persiapan Fitur untuk MLlib

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Assemble fitur ke vector
assembler = VectorAssembler(
    inputCols=["frequency", "avg_monetary", "avg_quantity"],
    outputCol="features_raw"
)
df_assembled = assembler.transform(df_customer)

# Standarisasi fitur (penting untuk K-Means!)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("✅ Fitur di-assemble dan di-scale")
df_scaled.select("Customer_ID", "features").show(5, truncate=False)

## 6.4 Menentukan Jumlah Cluster Optimal (Elbow Method)

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Test k = 2 sampai 8
costs = []
silhouettes = []
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

print("=== ELBOW METHOD ===")
print(f"{'k':>3} | {'Cost (WSSSE)':>15} | {'Silhouette':>10}")
print("-" * 40)

for k in range(2, 9):
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="prediction")
    model = kmeans.fit(df_scaled)
    cost = model.summary.trainingCost
    
    predictions = model.transform(df_scaled)
    silhouette = evaluator.evaluate(predictions)
    
    costs.append(cost)
    silhouettes.append(silhouette)
    print(f"{k:>3} | {cost:>15.2f} | {silhouette:>10.4f}")

print("\n📌 Pilih k di mana cost mulai landai (elbow) dan silhouette tinggi")

## 6.5 Train K-Means (k=3)

In [ ]:
# Final model dengan k=3
kmeans_final = KMeans(
    k=3, 
    seed=42, 
    featuresCol="features", 
    predictionCol="segment"
)
model_final = kmeans_final.fit(df_scaled)

# Transform
df_segmented = model_final.transform(df_scaled)
print("✅ K-Means training selesai (k=3)")
print(f"   WSSSE (Cost): {model_final.summary.trainingCost:.2f}")

## 6.6 Analisis Hasil Segmentasi

In [ ]:
print("=== PROFIL SETIAP SEGMEN ===\n")

df_segmented.groupBy("segment").agg(
    count("*").alias("jumlah_pelanggan"),
    avg("avg_monetary").alias("avg_spending"),
    avg("frequency").alias("avg_frequency"),
    avg("avg_quantity").alias("avg_quantity"),
    avg("total_spending").alias("avg_total_spending")
).orderBy("segment").show()

# Label segmen
print("📌 Interpretasi Segmen:")
print("   Segment dengan avg_spending tertinggi → High Value")
print("   Segment dengan avg_spending sedang    → Medium Value")  
print("   Segment dengan avg_spending terendah  → Low Value")

## 6.7 Distribusi Segmen

In [ ]:
print("=== DISTRIBUSI SEGMEN ===\n")
total = df_segmented.count()
for row in df_segmented.groupBy("segment").count().orderBy("segment").collect():
    pct = row["count"] / total * 100
    print(f"   Segment {row['segment']}: {row['count']} pelanggan ({pct:.1f}%)")

## 6.8 Simpan Hasil Segmentasi

In [ ]:
# Simpan ke Parquet
OUTPUT_PATH = "/output/customer_segments"

df_segmented.select(
    "Customer_ID", "frequency", "avg_monetary", 
    "avg_quantity", "total_spending", "segment"
).write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"✅ Hasil segmentasi disimpan ke: {OUTPUT_PATH}")